In [1]:
from typing import Callable
from os import makedirs
from os.path import join
import numpy as np
import pandas as pd
from verma_net_radiation_sensitivity.verma_net_radiation_sensitivity import process_verma_net_radiation_table
from PTJPL_sensitivity import process_PTJPL_table
from sensitivity import perturbed_run, sensitivity_analysis
import matplotlib.pyplot as plt

In [2]:
input_df = pd.read_csv("calval_final_50_PTJPL_inputs.csv")
input_df = input_df[input_df.fAPARmax != 0]
input_df

,Unnamed: 0.1,tower,lat,lon,orbit,scene,tile,time_UTC,date_UTC,doy,...,VISdiff,NIRdiff,VISdir,NIRdir,SWout,LWin,LWout,hour_of_day,Topt,fAPARmax
0,0,US-Ha2,42.5393,-72.1779,9254,9,18TYN,2020-02-22 00:00:00,2020-02-22,53,...,0.000000,0.000000,0.000000,0.000000,0.000000,223.482510,310.837408,19,0.0,0.5545
1,1,US-Ha2,42.5393,-72.1779,10150,9,18TYN,2020-04-20 00:00:00,2020-04-20,111,...,0.000000,0.000000,0.000000,0.000000,0.000000,249.470817,483.042043,19,0.0,0.5545
2,2,US-Ha2,42.5393,-72.1779,10352,9,18TYN,2020-05-03 00:00:00,2020-05-03,124,...,0.000000,0.000000,0.000000,0.000000,0.000000,293.358154,368.124775,19,0.0,0.5545
3,3,US-Ha2,42.5393,-72.1779,10684,7,18TYN,2020-05-24 00:00:00,2020-05-24,145,...,3.627162,3.519072,0.337294,0.020315,0.662989,NaN,385.565499,19,0.0,0.5545
4,4,US-Ha2,42.5393,-72.1779,11069,8,18TYN,2020-06-18 00:00:00,2020-06-18,170,...,7.418505,8.696527,1.935220,0.123967,2.254700,380.029433,454.438012,19,0.0,0.5545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,247,US-xUK,39.0404,-95.1921,11909,6,15SUD,2020-08-11 00:00:00,2020-08-11,224,...,14.372975,12.682295,3.003822,0.000000,5.351180,414.848071,439.377199,17,5.8,0.4664
248,248,US-xUK,39.0404,-95.1921,16668,8,15SUD,2021-06-14 00:00:00,2021-06-14,165,...,32.456092,15.308895,93.412739,127.317082,27.991697,NaN,405.251251,17,5.8,0.4664
249,249,US-xUK,39.0404,-95.1921,16810,8,15SUD,2021-06-23 00:00:00,2021-06-23,174,...,38.645010,27.577193,80.881654,98.257540,28.021802,NaN,388.004691,17,5.8,0.4664
250,250,US-xUK,39.0404,-95.1921,16810,8,15SUD,2021-06-23 00:00:00,2021-06-23,174,...,38.645010,27.577193,80.881654,98.257540,28.021802,400.532814,463.095778,17,5.8,0.4664


In [3]:
np.nanmin(input_df.fAPARmax)

0.3301

In [4]:
def process_verma_PTJPL_table(input_df: pd.DataFrame) -> pd.DataFrame:
    return process_PTJPL_table(process_verma_net_radiation_table(input_df))

In [5]:
input_df = pd.read_csv("calval_final_50_PTJPL_inputs.csv")

if "Ta" in input_df and "Ta_C" not in input_df:
    # input_df.rename({"Ta": "Ta_C"}, inplace=True)
    input_df["Ta_C"] = input_df["Ta"]

input_df = input_df[input_df.fAPARmax.apply(lambda fAPARmax: fAPARmax > 0.001)]
input_df = input_df[input_df.NDVI.apply(lambda NDVI: NDVI > 0.05)]

np.nanmin(input_df.ST_C), np.nanmax(input_df.ST_C)

(0.89, 49.33)

In [6]:
input_variables = ["NDVI"]
output_variables = ["LE"]

perturbation_df, sensitivity_metrics_df = sensitivity_analysis(
    input_df=input_df,
    input_variables=input_variables,
    output_variables=output_variables,
    forward_process=process_verma_PTJPL_table
)

sensitivity_metrics_df

fwet: [1.         0.15793057 0.44113834        nan 0.09602198 0.08958605
 0.58537641 0.11928378 0.10542303        nan 0.37214264 0.11412145
 0.11412145 0.0850722         nan 0.12686642 0.04759573 0.14373841
 0.02821032 0.04151831 0.0197513  0.02433621        nan 0.09574094
        nan        nan 0.04653393 0.01745041        nan 0.0380222
 0.05839815 0.05839815 0.10172011        nan 0.10563199 0.04631008
        nan 0.29366933 0.17201524 0.12446035 0.12446035 0.17198672
        nan        nan        nan 0.47679209        nan        nan
        nan 0.80149158        nan        nan 0.05703382 0.05703382
 0.36380487 0.15570133 0.13986969 0.15576457 0.15169591 0.10215691
 0.10425186 0.07868141 0.07868141 0.07985754 0.07985754 0.07505535
 0.07505535 0.08032654 0.08032654        nan        nan 0.10325623
 0.10325623 0.10325623 0.12063    0.12063    0.12063    0.07396403
 0.07396403 0.07396403 0.0899355  0.0899355  0.0899355         nan
        nan        nan        nan        nan        nan 0

TypeError: unsupported operand type(s) for -: 'float' and 'NoneType'

In [ ]:
input_variable = "LST"
output_variable = "LE"
results = perturbed_run(input_df, input_variable, output_variable, process_verma_PTJPL_table)
results

In [ ]:
model_name = "PT-JPL"
forward_process = process_verma_PTJPL_table   
input_variables = ["ST_C", "NDVI", "albedo", "Ta_C", "RH", "Rg"]
output_variables = ["LE", "LE_canopy", "LE_interception", "LE_soil"]

results = []
correlations = []

for input_variable in input_variables:
    for output_variable in output_variables:
        print(f"input variable: {input_variable}")
        print(f"output variable: {output_variable}")
        
        # run forward process with perturbation
        run_results = perturbed_run(input_df, input_variable, output_variable, forward_process)
        run_results = run_results[run_results[f"{output_variable}_perturbed"] != 0]
        run_results = run_results[run_results[f"{output_variable}_perturbation"] != 0]

        if len(run_results) == 0:
            print(f"no relationship between {input_variable} and {output_variable}")
            continue

        directory = join("analysis_CSVs", model_name)
        makedirs(directory, exist_ok=True)
        filename = join(directory, f"{model_name}_{input_variable}_to_{output_variable}.csv")
        print(filename)
        run_results.to_csv(filename, index=False)

        # input_perturbation = np.array(run_results[f"{input_variable}_perturbation_std"])
        input_perturbation = np.array(run_results[f"{input_variable}_perturbation"])
        # print(len(input_perturbation))
        # output_perturbation = np.array(run_results[f"{output_variable}_perturbation_std"])
        output_perturbation = np.array(run_results[f"{output_variable}_perturbation"])
        # print(len(output_perturbation))
        # perturbation_correlation = np.corrcoef(input_perturbation, output_perturbation)[0,1]
        # correlations.append([input_variable, output_variable, perturation_core])

        # create figure and axis objects
        fig, ax = plt.subplots()

        # create scatter plot
        ax.scatter(input_perturbation, output_perturbation)

        # set title and axis labels
        ax.set_title(f"{model_name} {input_variable} to {output_variable} Sensitivity")
        # ax.set_xlabel(f"{input_variable} Input Perturbation Sigma")
        # ax.set_ylabel(f"{output_variable} Output Perturbation Sigma")
        ax.set_xlabel(f"{input_variable} Input Perturbation")
        ax.set_ylabel(f"{output_variable} Output Perturbation")
        # show plot
        plt.show()

        results.append(run_results)

results = pd.concat(results)
# correlations = pd.DataFrame(correlations, columns=["input_variable", "output_variable", "correlation"]) 
# correlations
results

In [ ]:
results_std = results[[column for column in results.columns if column.endswith("std")]]
results_std.columns = [column.replace("_perturbation_std", "") for column in results_std.columns]
results_std_corr = results_std.corr().round(2)
results_std_corr

In [9]:
results_std_corr.to_csv('results_std_corr.csv', index=True)